## Abstract

The arXiv is the leading open-access repository for research preprints in fields such as Physics, Mathematics, and Computer Science. With a growing volume of daily submissions, the visibility and audience of individual preprints can vary significantly depending on posting dynamics. This project aims to develop a forecasting algorithm to predict arXiv posting behavior per category, with the following goal in mind:  
- **Short-term forecasting** for optimizing a preprint's *visibility*  

By analyzing historical arXiv posting data and usage statistics, we build models that can assist researchers in strategically choosing submission dates to increase their paper's exposure and readership.


## Stakeholders and KPIs

- **Primary Stakeholders:** Researchers who submit preprints to the arXiv and seek to optimize visibility and impact.  
- **Secondary Stakeholders:** arXiv moderators, research institutions, and bibliometric analysts interested in patterns of scientific communication.

### Short-term Forecasting Goal
To help researchers post on days when fewer papers are submitted in their category, thereby improving visibility.

### Key Assumption
The probability of a preprint being viewed is inversely proportional to the number of other preprints in the same category on the same day.

### Key Performance Indicator (KPI)
The proposed forecasting method should enable authors to post on days when the expected number of competing preprints is significantly lower than the average for that category.


## Cleaning and Preprocessing

### Pre-cleaning

To reduce the size and improve the usability of the dataset, we begin by stripping unnecessary content:
- We drop the articles with **no abstract** from each record and save the new dataset to `data/arxiv-metadata-noabstract.parquet`.
- Next, we extract only the columns: `id`, `versions`, and `categories`, and save the resulting lighter dataset as `data/arxiv-metadata-id-versions-categories.parquet`.

---

### Extracting the Date of First Version (v1)

We choose the date of the **first version (`v1`)** of each preprint as the canonical date of submission, because:
- The `update_date` field is unreliable (all papers show an update in May 2007).
- The first version is consistently present and accessible.

To obtain this:
- We define a `date_extractor` function to extract the first date from the list in the `versions` column.
- Apply this function to create a new column `date`.
- Drop the `versions` column.
- Save the resulting dataset to `data/arxiv-id-date-categories.parquet`.

---

### Cleaning the Categories

The `categories` field contains space-separated strings listing all arXiv categories a preprint belongs to. This needs cleaning because:
- The list contains **obsolete or deprecated category names**.
- Some historical categories were **later split** into more specific subfields (e.g., `q-bio`, `cond-mat`, `astro-ph`).

**Steps taken:**
- Import the current list of valid arXiv categories into `arxiv_categories`.
- Split category strings into lists.
- Identify and store **deprecated or unrecognized categories** in `missing_categories`.
- Define a **mapping dictionary `cat_dictionary`** that translates old category names into current ones (including meta-categories for pre-division entries).
- Apply a `translate()` function to clean up each category list using this mapping, while removing duplicates.

Finally, the cleaned dataset is saved as:


## Data Crunching

### Objective

This stage transforms the cleaned arXiv metadata into structured, analysis-ready dataframes that track both:
- Daily cross-listings between arXiv categories.
- Total number of listings per category per day.

These processed outputs are foundational for time series modeling and forecasting.

### Inputs
- `arxiv-metadata-cleaned.parquet`: Cleaned metadata including post dates and categories.
- `arxiv-categories.json`: List of current arXiv categories.

### Outputs
- `arxiv-snapshots.parquet`: A dataframe indexed by `date`, detailing cross-listings between category pairs.
- `arxiv-totals.parquet`: A dataframe indexed by `date`, containing total listing counts for each category.

### Cross-listings Construction

To capture the structure of how papers are posted in multiple categories:
- A list of category tuples (with repetitions) is built to represent possible category co-occurrences.
- A multi-index is created using these tuples.
- A function `take_snapshot` maps a day’s listings to a dictionary of cross-listings.
- Using `groupby` on `date`, the cross-listing structure is applied daily and saved.

### Totals per Category Construction

To compute daily posting volume in each category:
- A function `take_totals` aggregates daily counts per category.
- Data is grouped by `update_date`, exploded by category, and reindexed by `date`.
- Missing category-day entries are filled with 0 (no postings).
- The result is a dense daily timeseries of postings per category.

Both dataframes are chronologically sorted and saved for downstream modeling.

### Summary

The `crunching.ipynb` notebook builds the essential time-indexed structures — cross-listings and total counts — that power the forecasting component of the project. These datasets are saved in a compact, optimized format to facilitate rapid access and analysis in subsequent notebooks.


## Usage Data Summary

We analyze the arXiv usage logs from `arxiv-usage.parquet`, which records the number of daily connections to the arXiv website between 2024-01-01 and 2025-04-10.

### Key Insights:
- **Weekly pattern**: Connections tend to drop significantly on weekends, with Monday and Tuesday seeing the highest traffic.
- **Daily variation**: There is notable variance even among weekdays, suggesting that user activity is influenced by more than just calendar effects.

### Hypothesis Testing:
We tested whether the **day of the week** contributes significantly to the number of connections using an F-test between a reduced model (intercept only) and a full model (with one-hot encoded weekday variables).  
- **F-test p-value**: 0.706  
- **Conclusion**: No statistically significant evidence that weekday explains variation in arXiv traffic. The null hypothesis (average + noise) is not rejected.

This analysis informs the short-term forecasting goals by identifying when visibility might be highest based on user engagement patterns.


# Modeling Approach Summary

## Goal
To develop a short-term forecasting algorithm that suggests the optimal submission date (within a given time window) for a paper, with the aim of maximizing its visibility on arXiv.

## Stakeholders
Researchers planning to submit their preprints and aiming to avoid days with high volume in their submission category.

## Forecasting Strategy
The system identifies the best day(s) within a specified horizon \( h \) (number of business days) to submit a paper tagged with a set of arXiv categories \( T \).

## Step 1: Data Preparation
- Data source: arXiv metadata (post-cleaning).
- Time range: January 1, 2001 to March 17, 2025.
- Focus: Single or multiple arXiv categories.
- Exclude: Weekends (and potentially holidays).

## Step 2: Baseline Models
Implemented time-series models to forecast daily submission counts:
- **Holt-Winters (Triple Exponential Smoothing)**:
  - Additive and multiplicative versions tested.
  - Tuned hyperparameters: trend, damped trend, and seasonality.
- **SARIMA / ARIMA**:
  - Captures autoregressive and moving average structure in submissions.
- **Linear Regression with Time Features**:
  - Leveraged temporal variables like day of week, trend, and lag features.

## Step 3: Model Extensions
Explored more flexible and modular tools:
- **Facebook Prophet**:
  - Handles holidays, changepoints, and daily/weekly/yearly seasonality.
- **NeuralProphet**:
  - Allows human-in-the-loop modeling and iterative refinements.

These tools enable integration of exogenous signals (e.g., web usage patterns or long-term category trends).

## Step 4: Recommendation Framework
Given tags \(T\) and forecast horizon \(h\):
1. Assign a relevance weight \(w_c\) for each category \(c\).
2. Run the short-term forecast for each category over the horizon.
3. Score each day based on the weighted expected submission load.
4. Suggest the best submission day(s) with the lowest expected competition.

### Long-Term Integration (Optional)
- Predict longer-term trends in each category.
- Use long-term forecasts to adjust short-term weights or scoring metrics.
- Enables robust planning and broader audience targeting.



# Baseline Models Summary 

## Linear Regression with Time Features

## Objective
To identify a reliable short-term forecasting model for the number of daily arXiv submissions in individual categories, using simple regression-based baselines.

## Data
- Source: `data/arxiv-totals.parquet`
- Timeframe: January 1, 2001 – March 14, 2025 (training), March 17, 2025 onward (testing)
- Focus: Predict 5-day submission volume using the previous 150 days

## Preprocessing
- Removed early noisy data prior to 2001.
- Extracted day-of-week and applied one-hot encoding for use in regression.
- Analyzed seasonality using autocorrelation and partial autocorrelation plots, which showed strong weekly cycles in categories like `math` and `hep-ph`.

## Models Compared
Four baseline models were tested using 5-fold time series cross-validation:
1. **Dummy**: Predicts the mean of the past 150 days.
2. **Time Regression (t_reg)**: Linear regression on time only.
3. **Day-of-Week Regression (day_reg)**: Linear regression on weekday (categorical).
4. **Time + Day Regression (tday_reg)**: Linear regression on both time and weekday.

## Results
| Model        | Mean Absolute Percentage Error |
|--------------|-------------------------------|
| Dummy        | 1.475                          |
| Time         | 1.512                          |
| Weekday      | **1.378** (best)              |
| Time + Weekday | 1.417                        |

- **Best performer**: `day_reg` (weekday-only regression).
- **Best in class for 53.5% of categories**.
- **Max improvement over dummy**: 0.65
- **Mean improvement over dummy**: 0.13

## Output
- Saved model comparison results to `baseline-models-comparison.csv`.

## Conclusion
Regression on the day of the week consistently outperformed other baselines, making it a strong candidate for short-term visibility optimization.
